In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords

# add project root to path
sys.path.append(os.path.abspath("."))
from src.data import load_all_datasets

# download nltk data
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

# set aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.family"] = "sans-serif"
PRIMARY_COLOR = "#615fff"
SECONDARY_COLOR = "#1d293d"

In [ ]:
data_dir = "data/raw/"
df = load_all_datasets(data_dir)
print(f"total samples: {len(df):,}")

# check for missing values
missing = df.isnull().sum()
print("\nmissing values:")
print(missing)

# check for duplicates
duplicates = df.duplicated(subset=['text']).sum()
print(f"\nduplicate texts: {duplicates}")

df.head()

In [ ]:
exploded = df.explode("labels")
split_dist = pd.crosstab(exploded['labels'], exploded['source'], normalize='columns') * 100

split_dist.plot(kind='barh', figsize=(12, 8), color=[SECONDARY_COLOR, PRIMARY_COLOR, "#94a3b8"])
plt.title("label distribution percentage by split", fontsize=14)
plt.xlabel("percentage (%)")
plt.ylabel("")
plt.legend(title="split")
plt.show()

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
label_data = mlb.fit_transform(df['labels'])
label_df = pd.DataFrame(label_data, columns=mlb.classes_)

co_occurrence = label_df.T.dot(label_df)
np.fill_diagonal(co_occurrence.values, 0)

plt.figure(figsize=(10, 8))
sns.heatmap(co_occurrence, annot=True, fmt="d", cmap="Blues")
plt.title("label co-occurrence matrix", fontsize=14)
plt.show()

In [ ]:
df["word_count"] = df["text"].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x="word_count", y="source", palette="muted")
plt.title("word count distribution by split")
plt.xlabel("words per sample")
plt.ylabel("split")
plt.show()

print("word count statistics:")
print(df.groupby('source')['word_count'].describe())

In [ ]:
def get_top_ngrams(corpus, n=1, top_k=10):
    words = [w.lower() for text in corpus for w in str(text).split() if w.lower() not in stop_words and w.isalnum()]
    if n > 1:
        # simple bigram logic
        words = [f"{words[i]} {words[i+1]}" for i in range(len(words)-1)]
    return Counter(words).most_common(top_k)

target_fallacies = ["appeal_to_authority", "slippery_slope", "false_dilemma"]

fig, axes = plt.subplots(1, len(target_fallacies), figsize=(18, 5))
for i, fallacy in enumerate(target_fallacies):
    subset = df[df['labels'].apply(lambda x: fallacy in x)]['text']
    top_words = get_top_ngrams(subset, n=1)

    words, counts = zip(*top_words) if top_words else ([], [])
    sns.barplot(x=list(counts), y=list(words), ax=axes[i], color=PRIMARY_COLOR)
    axes[i].set_title(f"top words: {fallacy.replace('_', ' ')}")

plt.tight_layout()
plt.show()

In [ ]:
all_text = " ".join(df['text'].values)
wordcloud = WordCloud(width=1000, height=500, background_color='white',
                      colormap='viridis', stopwords=stop_words).generate(all_text)

plt.figure(figsize=(15, 8))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.title("corpus word cloud", fontsize=16)
plt.show()

In [ ]:
df["num_labels"] = df["labels"].apply(len)
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x="num_labels", palette="magma")
plt.title("number of fallacy labels per sample")
plt.xlabel("count of labels")
plt.ylabel("frequency")

for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', fontsize=11, color='black', xytext=(0, 5),
                textcoords='offset points')

plt.show()

In [ ]:
has_context = df[df['context'].notna() & (df['context'] != 'None')]
if not has_context.empty:
    print(f"samples with context: {len(has_context)}")
    for i, (_, row) in enumerate(has_context.sample(min(2, len(has_context))).iterrows()):
        print(f"\n--- sample {i+1} ---")
        print(f"labels : {row['labels']}")
        print(f"context: {row['context'][:150]}...")
        print(f"text   : {row['text']}")
else:
    print("no context data found in this dataset.")